In [2]:
import math

In [3]:
# CDF - Value to Probability
def normal_cdf(x, mu=0, sigma=1):
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

# Exam scores are normal with μ = 50, σ = 5. What’s P(score ≤ 60)?
cdf_60 = normal_cdf(60, 50, 5)
print(f"     P(score ≤ 60) : {cdf_60}") # Ans: 0.977, 97.7% of scores fall at or below 60

# Range between 60 and 45
cdf_45_to_60 = normal_cdf(60, 50, 5) - normal_cdf(45, 50, 5)
print(f"P(45 ≤ score ≤ 60) : {cdf_45_to_60}") # Ans: 0.8186, 81.86% of scores are between 45 and 60

     P(score ≤ 60) : 0.9772498680518208
P(45 ≤ score ≤ 60) : 0.8185946141203637


In [4]:
# Inverse CDF - Probability to Value
def inverse_normal_cdf(p, mu=0, sigma=1, tolerance=0.00001):
    # if not standard, compute standard and rescale
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)

    low_z = -10.0   # normal_cdf(-10) is (very close to) 0
    hi_z  =  10.0   # normal_cdf(10) is (very close to) 1
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2
        mid_p = normal_cdf(mid_z)
        if mid_p < p:
            low_z = mid_z    # midpoint too low, search above it
        else:
            hi_z = mid_z     # midpoint too high, search below it

    return mid_z

# 0.0, the median of the standard normal
print(f"Standard Normal Median: {inverse_normal_cdf(0.5)}")

# Inverse of the scores
print(f"x for 0.9772498680518208: {inverse_normal_cdf(0.977, 50, 5)}")

Standard Normal Median: -9.5367431640625e-06
x for 0.9772498680518208: 59.977006912231445


In [5]:
# Binomial

from typing import Tuple

def normal_approximation_to_binomial(n: int, p: float) -> Tuple[float, float]:
    """Returns mu na sigma correcpsonding to a Binomial(n,p)"""
    mu = p * n # Mean
    sigma = math.sqrt(n * p * (1-p)) # SD

    return mu, sigma

print(f"mu and sigma: {normal_approximation_to_binomial(5, 0.5)}")

mu and sigma: (2.5, 1.118033988749895)


In [6]:
# The normal cdf _is_ the probability the variable is below a threshold
normal_probability_below = normal_cdf

# It's above the threshold if it's not below the threshold
def normal_probability_above(
        lo: float,
        mu: float = 0,
        sigma: float = 1
    ) -> float:
    """The probability that an N(mu, sigma) is greater than lo."""
    return 1 - normal_cdf(lo, mu, sigma)

# It's between if it's less than hi, but not less than lo
def normal_probability_between(
        lo: float,
        hi: float,
        mu: float = 0,
        sigma: float = 1
    ) -> float:
    """The probability that an N(mu, sigma) is between lo and hi."""
    return normal_cdf(hi, mu, sigma) - normal_cdf(lo, mu, sigma)

# It's outside if it's not between
def normal_probability_outside(
        lo: float,
        hi: float,
        mu: float = 0,
        sigma: float = 1
    ) -> float:
    """The probability that an N(mu, sigma) is not between lo and hi."""
    return 1 - normal_probability_between(lo, hi, mu, sigma)

In [7]:
# Setup: exam scores, μ = 50, σ = 5

# Think of scores as normally distributed. Useful landmarks:

# 45 is 1 SD below the mean (z = −1)
# 55 is 1 SD above (z = +1)
# 40 and 60 are 2 SDs away (z = ∓2)

## normal_probability_below: CDF itself

print(normal_probability_below(60, 50, 5) )  # 0.9772
print(normal_probability_below(50, 50, 5))   # 0.5000
print("\n")

## normal_probability_above: 1 - CDF

print(normal_probability_above(60, 50, 5) )  # 0.0228
print(normal_probability_above(50, 50, 5))   # 0.5000 
print("\n")

## normal_probability_between: CDF(hi) - CDF(lo)

print(normal_probability_between(45, 55, 50, 5))   # 0.6827
print(normal_probability_between(40, 60, 50, 5))   # 0.9545
print(normal_probability_between(45, 60, 50, 5))   # 0.8186
print("\n")

## normal_probability_outside: 1 - between
## - both tails

print(normal_probability_outside(45, 55, 50, 5))   # 0.3173
print(normal_probability_outside(40, 60, 50, 5))   # 0.0455
print(normal_probability_outside(35, 65, 50, 5))   # 0.0027
print("\n")


0.9772498680518208
0.5


0.02275013194817921
0.5


0.6826894921370859
0.9544997361036416
0.8185946141203637


0.31731050786291415
0.04550026389635842
0.002699796063260207




In [8]:
# Standard normal version (μ = 0, σ = 1)
# Defaults apply, so you can skip μ and σ:

print(normal_probability_above(1.96))            # 0.0250
print(normal_probability_between(-1.96, 1.96))    # 0.9500
print(normal_probability_outside(-1.96, 1.96))    # 0.0500

# This is the classic 95% / 5% split used in hypothesis testing

0.024997895148220373
0.9500042097035593
0.049995790296440745


In [9]:
# INVERSE CDF

def normal_upper_bound(
        probability: float,
        mu: float = 0,
        sigma: float = 1
    ) -> float:
    """Returns the z for which P(Z <= z) = probability"""
    return inverse_normal_cdf(probability, mu, sigma)

def normal_lower_bound(
        probability: float,
        mu: float = 0,
        sigma: float = 1
    ) -> float:
    """Returns the z for which P(Z >= z) = probability"""
    return inverse_normal_cdf(1 - probability, mu, sigma)

def normal_two_sided_bounds(
        probability: float,
        mu: float = 0,
        sigma: float = 1
    ) -> Tuple[float, float]:
    """
    Returns the symmetric (about the mean) bounds
    that contain the specified probability
    """
    tail_probability = (1 - probability) / 2
    # upper bound should have tail_probability above it
    upper_bound = normal_lower_bound(tail_probability, mu, sigma)
    # lower bound should have tail_probability below it
    lower_bound = normal_upper_bound(tail_probability, mu, sigma)
    return lower_bound, upper_bound


In [10]:
# 

print(normal_upper_bound(0.9772498680518208, 50, 5)) # 59.99
print(normal_lower_bound(0.02275013194817921, 50, 5)) # 59.99
print(normal_two_sided_bounds(0.9772498680518208, 50, 5)) # 

59.999990463256836
59.999990463256836
(38.61193656921387, 61.38806343078613)


In [ ]:
# COIN FLIP

# n = 1000
# mu = 500
# sd = 15.8

mu_0, sigma_0 = normal_approximation_to_binomial(1000, 0.5)
print(f"mu: {mu_0}, \nsigma: {sigma_0}\n")

# Type 1 error of a false positive, reject Ho even though its true
# set at 5% error rate

# Consider test rejects Ho if X falls outside the bounds given by:
lower_bound, upper_bound = normal_two_sided_bounds(0.95, mu_0, sigma_0)
print(f"lower_bound: {lower_bound}, \nupper_bound: {upper_bound}\n")

# Assuming that p = 0.5 (Ho is true) then there is a 5% chance that we observe that X lies outside the interval  

mu: 500.0, 
sigma: 15.811388300841896

lower_bound: 469.01026640487555, 
upper_bound: 530.9897335951244



In [13]:
# Type 2 error - False negative - Probability of fail to reject Ho even though its false

# 95% bounds based on assumption p is 0.5
lo, hi = normal_two_sided_bounds(0.95, mu_0, sigma_0)

# actual mu and sigma based on p = 0.55, biased towards heads
mu_1, sigma_1 = normal_approximation_to_binomial(1000, 0.55)

# a type 2 error means we fail to reject the null hypothesis,
# which will happen when X is still in our original interval
type_2_probability = normal_probability_between(lo, hi, mu_1, sigma_1)
power = 1 - type_2_probability # 0.887
print(f"Power1: {power}\n")

# When coin is not biased towards head or that p ≤ 0. 5
# In that case we want a one-sided test that rejects the null hypothesis when X is much larger than 500 but not when X is smaller than 500.
# So, a 5% significance test involves using normal_probability_below to find the cutoff below which 95% of the probability lies:

hi = normal_upper_bound(0.95, mu_0, sigma_0)
# is 526 (< 531, since we need more probability in the upper tail)

type_2_probability = normal_probability_below(hi, mu_1, sigma_1)
power = 1 - type_2_probability # 0.936
print(f"Power2: {power}\n")




Power1: 0.8865480012953671

Power2: 0.9363794803307173



In [ ]:
# P-Values

# - Instead of choosing bounds based on probability cutoffs, we compute probability - assuming Ho is true - that we would see a value at least as extreme as the one we actually observered  

def two_sided_p_value(x: float, mu: float = 0, sigma: float = 1) -> float:
    """
    How likely are we to see a value at least as extreme as x (in either
    direction) if our values are from an N(mu, sigma)?
    """
    if x >= mu:
        # x is greater than the mean, so the tail is everything greater than x
        return 2 * normal_probability_above(x, mu, sigma)
    else:
        # x is less than the mean, so the tail is everything less than x
        return 2 * normal_probability_below(x, mu, sigma)


print(f"{two_sided_p_value(529.5, mu_0, sigma_0)}") # 0.062

0.06207721579598835


In [23]:
import random

extreme_value_count = 0
for _ in range(1000):
    num_heads = sum(1 if random.random() < 0.5 else 0    # Count # of heads
        for _ in range(1000))                            # in 1000 flips,
    if num_heads >= 530 or num_heads <= 470:             # and count how often
        extreme_value_count += 1                         # the # is 'extreme'

    print(f"No. Heads: {num_heads}, extreme: {extreme_value_count}")

# p-value was 0.062 => ~62 extreme values out of 1000
assert 59 < extreme_value_count < 65, f"{extreme_value_count}"

No. Heads: 492, extreme: 0
No. Heads: 511, extreme: 0
No. Heads: 490, extreme: 0
No. Heads: 499, extreme: 0
No. Heads: 508, extreme: 0
No. Heads: 480, extreme: 0
No. Heads: 511, extreme: 0
No. Heads: 505, extreme: 0
No. Heads: 501, extreme: 0
No. Heads: 513, extreme: 0
No. Heads: 499, extreme: 0
No. Heads: 499, extreme: 0
No. Heads: 502, extreme: 0
No. Heads: 512, extreme: 0
No. Heads: 506, extreme: 0
No. Heads: 482, extreme: 0
No. Heads: 523, extreme: 0
No. Heads: 495, extreme: 0
No. Heads: 508, extreme: 0
No. Heads: 477, extreme: 0
No. Heads: 495, extreme: 0
No. Heads: 512, extreme: 0
No. Heads: 496, extreme: 0
No. Heads: 513, extreme: 0
No. Heads: 493, extreme: 0
No. Heads: 479, extreme: 0
No. Heads: 504, extreme: 0
No. Heads: 488, extreme: 0
No. Heads: 480, extreme: 0
No. Heads: 483, extreme: 0
No. Heads: 508, extreme: 0
No. Heads: 480, extreme: 0
No. Heads: 483, extreme: 0
No. Heads: 511, extreme: 0
No. Heads: 485, extreme: 0
No. Heads: 486, extreme: 0
No. Heads: 494, extreme: 0
N

AssertionError: 73